# Notebook 01: The LLM Landscape

This notebook compares major LLM providers, examines pricing, and demonstrates calling different models.

**Learning Objectives:**
- Compare major LLM providers (OpenAI, Google, Anthropic, Meta)
- Analyze API pricing structures
- Call different models via APIs
- Determine when to use which model

## 1. Major LLM Providers Overview

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Provider comparison table
providers = pd.DataFrame({
    "Provider": ["OpenAI", "Google", "Anthropic", "Meta", "Mistral AI", "DeepSeek"],
    "Flagship Model": ["GPT-4o", "GPT-4o", "Claude Opus 4", "Llama 3.1", "Mistral Large", "DeepSeek-V3"],
    "Context Window": ["128K", "1M+", "200K", "128K", "128K", "128K"],
    "Type": ["Proprietary", "Proprietary", "Proprietary", "Open-Weight", "Open-Weight", "Open-Weight"],
    "Strengths": [
        "Broad knowledge, tool use, fast",
        "Massive context, multimodal",
        "Long-context, safety, reasoning",
        "Community, fine-tuning, self-host",
        "Efficient, multilingual",
        "MoE efficiency, strong reasoning"
    ]
})

display(providers.style.set_properties(**{'text-align': 'left'}))

## 2. API Pricing Comparison

Pricing as of early 2025. Always verify current pricing on provider websites.

In [ ]:
pricing = pd.DataFrame({
    "Model": [
        "GPT-4o", "GPT-4o-mini",
        "GPT-4o", "GPT-4o Mini",
        "Claude Opus 4", "Claude Sonnet 4", "Claude Haiku 3.5",
        "Llama 3.1 405B (hosted)", "Llama 3.1 8B (hosted)",
        "Mistral Large", "Mistral Small",
        "DeepSeek-V3"
    ],
    "Input Cost ($/1M tokens)": [
        2.50, 0.15,
        1.25, 0.15,
        15.00, 3.00, 0.80,
        3.00, 0.10,
        2.00, 0.10,
        0.27
    ],
    "Output Cost ($/1M tokens)": [
        10.00, 0.60,
        10.00, 0.60,
        75.00, 15.00, 4.00,
        3.00, 0.10,
        6.00, 0.30,
        1.10
    ],
    "Tier": [
        "Premium", "Budget",
        "Premium", "Budget",
        "Premium", "Mid", "Budget",
        "Mid", "Budget",
        "Mid", "Budget",
        "Budget"
    ]
})

# Calculate cost per 1M tokens (input + output average)
pricing["Avg Cost/1M tokens"] = (pricing["Input Cost ($/1M tokens)"] + pricing["Output Cost ($/1M tokens)"]) / 2

display(pricing.style.format({
    "Input Cost ($/1M tokens)": "${:.2f}",
    "Output Cost ($/1M tokens)": "${:.2f}",
    "Avg Cost/1M tokens": "${:.2f}"
}).bar(subset=["Avg Cost/1M tokens"], color="#5fba7d")

### Cost Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Input costs
models = pricing["Model"]
x = np.arange(len(models))
width = 0.35

axes[0].barh(x, pricing["Input Cost ($/1M tokens)"], width, label="Input", color="#4CAF50")
axes[0].set_xlabel("Cost per 1M Tokens ($)")
axes[0].set_title("Input Pricing")
axes[0].set_yticks(x)
axes[0].set_yticklabels(models, fontsize=8)
axes[0].legend()

# Output costs
axes[1].barh(x, pricing["Output Cost ($/1M tokens)"], width, label="Output", color="#f44336")
axes[1].set_xlabel("Cost per 1M Tokens ($)")
axes[1].set_title("Output Pricing")
axes[1].set_yticks(x)
axes[1].set_yticklabels(models, fontsize=8)
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Calling Models via API

Below we demonstrate calling OpenAI and Anthropic APIs. Replace with your actual API keys.

**Note:** If you don't have API keys, you can still follow along by reading the code and expected outputs.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

print(f"OpenAI key: {'Set' if OPENAI_KEY else 'Not set'}")
print(f"Anthropic key: {'Set' if ANTHROPIC_KEY else 'Not set'}")

### 3.1 OpenAI API Call

In [ ]:
from openai import OpenAI

if OPENAI_KEY:
    client = OpenAI(api_key=OPENAI_KEY)
    
    prompt = "Explain the difference between a stack and a queue in 2 sentences."
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=150
    )
    
    print(f"Model: {response.model}")
    print(f"Response: {response.choices[0].message.content}")
    print(f"Tokens used: {response.usage.total_tokens}")
    print(f"Cost estimate: ~${response.usage.total_tokens * 0.0000006:.6f}")
else:
    print("OpenAI API key not set. Add it to .env file.")
    print("\nExpected output:")
    print("A stack is a LIFO (Last In, First Out) structure where elements are added and removed from the top.")
    print("A queue is a FIFO (First In, First Out) structure where elements are added at the back and removed from the front.")

### 3.2 Anthropic API Call

In [ ]:
import anthropic

if ANTHROPIC_KEY:
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    
    prompt = "Explain the difference between a stack and a queue in 2 sentences."
    
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    )
    
    print(f"Model: {response.model}")
    print(f"Response: {response.content[0].text}")
    print(f"Tokens used: input={response.usage.input_tokens}, output={response.usage.output_tokens}")
else:
    print("Anthropic API key not set. Add it to .env file.")
    print("\nExpected output:")
    print("A stack follows Last-In-First-Out (LIFO) ordering, like a pile of plates.")
    print("A queue follows First-In-First-Out (FIFO) ordering, like a line at a store.")

## 4. Side-by-Side Model Comparison

Send the same prompt to multiple models and compare outputs.

In [ ]:
test_prompts = [
    "What are the SOLID principles in software engineering?",
    "Write a Python function to find the longest common subsequence of two strings.",
    "Explain quantum entanglement to a 10-year-old."
]

results = []

# Collect results from available APIs
for prompt in test_prompts:
    row = {"Prompt": prompt[:60] + "..."}
    
    # OpenAI
    if OPENAI_KEY:
        try:
            resp = client_openai.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=300
            )
            row["GPT-4o-mini"] = resp.choices[0].message.content[:200]
            row["GPT-4o-mini Tokens"] = resp.usage.total_tokens
        except Exception as e:
            row["GPT-4o-mini"] = f"Error: {e}"
    else:
        row["GPT-4o-mini"] = "(API key not set)"
    
    results.append(row)

if results:
    display(pd.DataFrame(results))
else:
    print("No API keys configured. Add keys to .env to run comparisons.")

## 5. When to Use Which Model

### Decision Framework

In [ ]:
use_cases = pd.DataFrame({
    "Use Case": [
        "Quick classification",
        "Customer support chatbot",
        "Complex reasoning / analysis",
        "Code generation",
        "Document summarization",
        "Content at scale",
        "Fine-tuned domain-specific",
        "Privacy-sensitive deployment"
    ],
    "Recommended Model": [
        "GPT-4o-mini / Haiku 3.5",
        "GPT-4o-mini / Gemini Flash",
        "GPT-4o / Claude Sonnet 4",
        "GPT-4o / Claude Sonnet 4",
        "Claude Opus 4 / Gemini Pro",
        "GPT-4o-mini / DeepSeek-V3",
        "Llama 3.1 / Mistral (self-hosted)",
        "Llama 3.1 / Qwen (self-hosted)"
    ],
    "Why": [
        "Fast, cheap, good enough quality",
        "Good quality, low cost per conversation",
        "Best reasoning, worth the cost",
        "Strong code training, good at following specs",
        "Long context handles full documents",
        "Low cost at high volume",
        "Full fine-tuning control, no data sharing",
        "Data never leaves your infrastructure"
    ]
})

display(use_cases.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]}
]))

## 6. Cost Estimation Calculator

In [ ]:
def estimate_monthly_cost(
    daily_conversations: int,
    messages_per_conversation: int,
    avg_input_tokens: int,
    avg_output_tokens: int,
    input_price_per_m: float,
    output_price_per_m: float,
    days_per_month: int = 30
) -> dict:
    """Estimate monthly API cost for a chatbot application."""
    total_input_tokens = daily_conversations * messages_per_conversation * avg_input_tokens * days_per_month
    total_output_tokens = daily_conversations * messages_per_conversation * avg_output_tokens * days_per_month
    
    input_cost = (total_input_tokens / 1_000_000) * input_price_per_m
    output_cost = (total_output_tokens / 1_000_000) * output_price_per_m
    total_cost = input_cost + output_cost
    
    return {
        "Monthly Input Tokens": f"{total_input_tokens:,.0f}",
        "Monthly Output Tokens": f"{total_output_tokens:,.0f}",
        "Input Cost": f"${input_cost:.2f}",
        "Output Cost": f"${output_cost:.2f}",
        "Total Monthly Cost": f"${total_cost:.2f}",
        "Cost per Conversation": f"${total_cost / (daily_conversations * days_per_month):.4f}"
    }

# Example: customer support chatbot
print("=== Customer Support Chatbot (5,000 conversations/day) ===")
print()

scenarios = [
    ("GPT-4o-mini", 0.15, 0.60),
    ("Claude Haiku 3.5", 0.80, 4.00),
    ("GPT-4o", 2.50, 10.00),
    ("DeepSeek-V3", 0.27, 1.10),
]

cost_comparison = []
for name, in_price, out_price in scenarios:
    result = estimate_monthly_cost(
        daily_conversations=5000,
        messages_per_conversation=10,
        avg_input_tokens=150,
        avg_output_tokens=200,
        input_price_per_m=in_price,
        output_price_per_m=out_price
    )
    result["Model"] = name
    cost_comparison.append(result)

display(pd.DataFrame(cost_comparison).set_index("Model"))

## Key Takeaways

1. **No single "best" model** — the right choice depends on your use case, budget, and requirements
2. **Budget models** (GPT-4o-mini, Haiku 3.5, Gemini Flash) are sufficient for many production tasks
3. **Premium models** (GPT-4o, Claude Opus, Gemini Pro) are worth it for complex reasoning and analysis
4. **Open-weight models** (Llama, Mistral, DeepSeek) offer cost savings at scale and full control
5. **Always benchmark on your actual data** — public benchmarks don't always reflect real-world performance